# Kwon et al. (2025), Parsing the Pulse: Decomposing Macroeconomic Sentiment with LLMs
### Decomposing Macroeconomic Sentiment with an LLM
**Summer Camp — Short empirical exercise**

---

This notebook walks through a small pipeline that reads news articles and turns them into **structured macroeconomic sentiment data**, reproducing the approach of:

> Kwon, B., T. Park, P. Rungcharoenkitkul, and F. Smets (2025): *"Parsing the pulse: decomposing macroeconomic sentiment with LLMs"*, BIS Working Paper No. 1294.

By the end you will understand both halves of the exercise:

1. **The economics**: what we are measuring (demand vs. supply drivers, sentiment, the detailed subtypes) and *why the classification is split into two stages*.
2. **The code mechanics**: how to send a PDF to the Claude API, force structured JSON out of a language model, chain two model calls, and assemble the results into the authors' spreadsheet format.

## The big picture
---
We are converting *unstructured text* (a news article) into a *number* the way an economist would code it by hand, but at scale and reproducibly.

For each article we want to end up with a sentiment score placed in the right conceptual bucket:

```
article ──▶ [ Stage 1: what is driving this, and is it good or bad? ]
                          │
                          ▼
            Driver = Demand / Supply / Both
            Sentiment = Positive (+1) / Negative (−1) / Neutral (0)
                          │
                          ▼
        ──▶ [ Stage 2: *what kind* of demand/supply driver is it? ]
                          │
                          ▼
            Subtype = Real-Private / Fiscal / Financial-Private /
                      Monetary / Commodity / Supply-disruption /
                      Gov-Policy / Other
                          │
                          ▼
        ──▶ a row in the spreadsheet: the score lands in the subtype column
```

The rest of the notebook builds this picture one block at a time.

## 1. The economic concepts

### 1.1 Why demand vs. supply?

The central organizing idea comes from textbook macro. A shock can move the economy from either side of the market, and the **two sides have opposite signatures in the data**:

| | Output / activity | Inflation |
|---|---|---|
| **Demand driver** (e.g. a tax cut, a confidence boom, rate cuts) | ↑ | ↑ (same direction) |
| **Supply driver** (e.g. an oil spike, a supply-chain break) | ↓ | ↑ (opposite direction) |

So if you can read an article and decide *which side* is dominant, you have learned something an aggregate price or output number alone cannot tell you: a rise in inflation driven by demand means something very different for policy than the same rise driven by supply.

That is the entire motivation for the paper, using a language model to read thousands of articles and tag each one's **driver** and **sentiment**, then aggregating into an index.

### 1.2 Sentiment

Within a chosen driver, the **sentiment** is the direction of the effect:

- **Positive** → expansion (faster growth, more hiring) for the *growth* task; higher inflationary pressure for the *inflation* task. Scored **+1**.
- **Negative** → contraction / lower inflation. Scored **−1**.
- **Neutral** → related but no clear direction, or offsetting forces. Scored **0**.
- **Null** → article not related to the topic at all.

The same article is read **twice**: once asking "what does this say about *growth*?" and once "what does this say about *inflation*?" The sign can differ — a supply shock is bad for growth (−1) but pushes inflation up (+1).

### 1.3 The detailed subtypes

Knowing "demand, positive" is useful; knowing it is *fiscal* demand vs. *monetary* demand is more useful. Stage 2 refines the driver into one of eight subtypes:

**Demand:** Real-Private · Real-Fiscal-Policy · Financial-Private · Financial-Monetary-Policy
**Supply:** Commodity prices · Supply disruptions · Government Policy · Others

These map one-to-one onto the columns of the authors' spreadsheet, which is why we keep them in exactly this vocabulary.

### 1.4 Why two stages instead of one big prompt?

You might ask the model to do all of this in a single call. The paper (and good practice) splits it for several reasons worth internalizing as a *method*, not just a coding trick:

1. **Decomposition improves accuracy.** A narrow question ("is this demand or supply?") is answered more reliably than a compound one. Each stage has a smaller, cleaner decision.
2. **The second question is conditional on the first.** "What *type* of demand driver?" only makes sense once you've established it *is* demand. Stage 2 receives Stage 1's answer as given context, so it isn't re-litigating the driver — it's refining it.
3. **Auditability.** When a classification looks wrong, you can see *which* stage erred. A single black-box call gives you no such handle.
4. **It mirrors how a human coder would work** first the coarse call, then the fine one.

This conditional structure (Stage 2 takes Stage 1's output as input) is the single most important design idea in the pipeline.

## 2. Setup

The API key is read from an **environment variable**, never written in the notebook. Hardcoding a key is the most common way people accidentally leak credentials (e.g. by committing the file). Set it *before* launching Jupyter:

- PowerShell: `$env:ANTHROPIC_API_KEY = "your-key"`
- macOS/Linux: `export ANTHROPIC_API_KEY="your-key"`

In [ ]:
# Run once if needed:
# %pip install anthropic openpyxl

import os
import re
import json
import base64
from pathlib import Path

from anthropic import Anthropic
from openpyxl import Workbook
from openpyxl.styles import Font

# The client picks up ANTHROPIC_API_KEY from the environment automatically,
# but we read it explicitly so a missing key fails loudly and early.
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-8"   # current model ID. Cheaper alternative: "claude-sonnet-4-6"
print("Client ready, using model:", MODEL)

---
> ### 🔧 YOUR TURN
> The pipeline uses `MODEL = "claude-opus-4-8"`. Opus is the most capable but also the most expensive model.
>
> 1. Change `MODEL` to `"claude-sonnet-4-6"` (faster and cheaper) and keep it for the rest of the notebook.
> 2. Later, after you've run a few articles on both, compare: do the *driver* and *sentiment* labels actually differ between the two models? Where do they disagree — the coarse Stage-1 call, or the finer Stage-2 subtype?
>
> *Why this matters:* picking a model is a cost/accuracy trade-off. For a coarse binary like demand-vs-supply a small model may be plenty; the conservative fiscal-policy call in Stage 2 is where a stronger model tends to earn its keep.
---

### 2.1 Which files are we processing?

The articles are PDFs in the same folder as the notebook. They are all from the same date, so for this exercise the date is just a shared label.

In [ ]:
PDF_DIR = Path(".")
PDF_FILES = [
    "ProQuestDocuments-1 (1).pdf",
    "ProQuestDocuments-1 (2).pdf",
    "ProQuestDocuments-1 (3).pdf",
    "ProQuestDocuments-1 (4).pdf",
]

RESULTS_JSON = "classification_results.json"
OUTPUT_XLSX  = "results_formatted.xlsx"
DATE_LABEL   = "03.06.2026"   # shared label for every row

# Sanity check: do the files exist where we expect?
for f in PDF_FILES:
    print(("found " if (PDF_DIR / f).exists() else "MISSING "), f)

## 3. Stage 1: the primary prompt (driver + sentiment)

### The economics, restated as instructions
The prompt below *is* the codebook. Everything from Section 1, the demand/supply definitions, the sentiment scale, the "both drivers" rule, the tie-breaking guidance for opposing sentiments, is written out so the model applies the same rules a trained human coder would.

### The code mechanics: forcing structured output
A language model returns free text by default. To get *data* we do two things:
1. Embed the **exact JSON schema** we want in the prompt and tell the model to return *only* that object.
2. Parse the reply back into a Python dict, defensively (models occasionally wrap JSON in markdown fences).

We write the growth and inflation prompts from shared building blocks so the two tasks stay consistent.

In [ ]:
PRIMARY_SCHEMA = """Output JSON format (rules embedded):
{
"Drivers of sentiment": "<Demand drivers|Supply drivers|Both drivers|Undeterminable>",
"Sentiment": {
  "Demand drivers": "<Positive|Negative|Neutral|Null> # Non-Null only if demand drivers are present",
  "Supply drivers": "<Positive|Negative|Neutral|Null> # Non-Null only if supply drivers are present"
},
"confidence": {
  "drivers": "<0.0-1.0>",
  "sentiment": "<0.0-1.0>"
},
"explanation": "<Concise summary of reasoning behind the classifications>"
}
Respond with ONLY the JSON object, no preamble, no markdown fences."""

_STEP1 = """Step 1. Classify the dominant drivers of the development:
- Demand drivers: drivers that change the desire to consume, spend, or invest, pushing activity and inflation in the SAME direction. Examples: consumer confidence, expected income, fiscal policy, financial/credit conditions, interest rates, monetary policy, exchange rates, liquidity.
- Supply drivers: drivers that affect production/output capacity, pushing activity and inflation in OPPOSITE directions. Examples: commodity prices, trade/tariff policy, subsidies, labour supply, structural reforms, supply-chain disruptions, natural disasters, productivity shocks.
- Both drivers: explicit evidence that demand and supply are equally dominant (sentiments may differ).
- Undeterminable: impossible to classify clearly."""

PRIMARY_PROMPTS = {
    "growth": f"""You are a macroeconomist specializing in U.S. economic trends. You will be given a news article related to U.S. macroeconomic activity. Analyze the text and (i) identify the key driver, then (ii) classify its sentiment.

{_STEP1}

Step 2. Classify sentiment (for each driver if 'Both drivers'):
- Positive: expansion (faster growth, rising incomes, more hiring, lower unemployment).
- Negative: contraction (slower growth, falling incomes, higher unemployment).
- Neutral: related but no clear direction, or offsetting/uncertain effects.
- Null: not related to macroeconomic activity.

Where the main development and its context conflict (e.g. a rate hike in response to strong demand), prioritise the sentiment of the development itself if macro-significant; otherwise use the broader context.

{PRIMARY_SCHEMA}""",

    "inflation": f"""You are a macroeconomist specializing in U.S. economic trends. You will be given a news article related to U.S. inflation development. Analyze the text and (i) identify the key driver, then (ii) classify its sentiment.

{_STEP1}

Step 2. Classify sentiment (for each driver if 'Both drivers'):
- Positive: higher inflationary pressure (faster CPI/PCE/PPI increases; consider delayed effects).
- Negative: lower inflationary pressure (slower increases or outright decreases).
- Neutral: related but no clear direction, or offsetting/uncertain effects.
- Null: not related to inflation.

{PRIMARY_SCHEMA}""",
}

print("Primary prompts built for:", list(PRIMARY_PROMPTS))

---
> ### 🔧 YOUR TURN
> The prompt **is** the codebook — the model only applies rules you write down.
>
> 1. Read the `growth` prompt in `PRIMARY_PROMPTS` closely. In your own words, what would the model do with an article about a one-off factory fire in a single town? Which category *should* that fall into, and is the prompt clear enough to get it there?
> 2. Add one sentence to the `_STEP1` block clarifying how to treat purely **local / one-firm** news (hint: it usually isn't a national macro driver). Rebuild the prompts by re-running the cell.
>
> *Why this matters:* reproducibility in LLM coding comes entirely from making the rules explicit. Vague rules → inconsistent labels across thousands of articles.
---

## 4. Stage 2: the subtype prompt (conditional refinement)

This prompt receives the article **plus Stage 1's verdict** ("the main driver is Demand, sentiment Positive") and answers the narrower question: *which* subtype?

Notice the prompt does not re-decide demand vs. supply — that is treated as settled input. This is the conditional structure from Section 1.4, made concrete. The detailed definitions (especially the deliberately *conservative* rule for calling something "fiscal policy") are the codebook for this finer decision.

In [ ]:
DRIVER_DEFS = """For demand drivers, choose:
- Real, private drivers: non-financial drivers (confidence, uncertainty, expected income) reflecting the private sector's willingness to spend. Unrelated to fiscal policy.
- Real, fiscal policy drivers: shifts in FEDERAL fiscal policy (new/revised legislation, proposed federal budget) that move aggregate demand via taxation or spending. Include shutdowns / legislative delays. Must be large enough to shift national output/inflation. EXCLUDE: reallocations of existing budgets; state/municipal policy (unless federally coordinated, nationally material); evaluations of past policy without new action; regulatory/trade/administrative rules. Be CONSERVATIVE — if ambiguous, it is NOT fiscal policy.
- Financial, private drivers: financial/credit conditions, cost of funds, rates, FX, liquidity, balance sheets — NOT directly induced by monetary policy.
- Financial, monetary policy drivers: financial conditions primarily driven by monetary policy (policy rate, asset purchases/sales, balance-sheet composition, or forward guidance). Do NOT answer this if monetary policy is not clearly the dominant factor among several.
- Undeterminable: cannot be classified; give one or two key words.

For supply drivers, choose:
- Commodity prices: oil, energy, metals, agricultural products, raw materials.
- Policy shifts: trade policy, tariffs, subsidies, structural reforms, other supply-side government policy.
- Supply disruptions: supply-chain problems, bottlenecks, global value-chain factors.
- Others: anything else (natural disasters, productivity shocks); give one or two key words."""

SUBTYPE_SCHEMA = """Output JSON format (rules embedded):
{
"Input context": {
  "Sentiment": "<Positive|Negative|Neutral>",
  "Main drivers": "<Demand drivers|Supply drivers>"
},
"Driver subtypes": {
  "Demand drivers": "<Real, private drivers|Real, fiscal policy drivers|Financial, private drivers|Financial, monetary policy drivers|Undeterminable|Null>",
  "Supply drivers": {
    "Type": "<Commodity prices|Policy shifts|Supply disruptions|Others|Null>",
    "Supply-Other Details": "<text if 'Others' else Null>"
  }
},
"confidence": { "subtypes": "<0.0-1.0>" },
"explanation": "<Concise reasoning>"
}
Respond with ONLY the JSON object, no preamble, no markdown fences."""

SUBTYPE_PROMPTS = {
    "growth": f"""You are a macroeconomist. You are given an article plus a sentiment about U.S. economic growth (positive/negative/neutral) and its main driver (demand or supply). Provide the detailed subtype.

{DRIVER_DEFS}

{SUBTYPE_SCHEMA}""",
    "inflation": f"""You are a macroeconomist. You are given an article plus a sentiment about U.S. inflation (positive = higher, negative = lower, neutral) and its main driver (demand or supply). Provide the detailed subtype.

{DRIVER_DEFS}

{SUBTYPE_SCHEMA}""",
}

print("Subtype prompts built for:", list(SUBTYPE_PROMPTS))

---
> ### 🔧 YOUR TURN
> Look at the **"Real, fiscal policy drivers"** definition. Notice how much text is spent telling the model what is *NOT* fiscal policy, and the instruction to be **conservative**.
>
> 1. Why do you think the authors made this category deliberately hard to assign? What kind of error are they trying to avoid?
> 2. Write down (in a new cell) one article headline that you think is genuinely fiscal-policy demand, and one that *looks* fiscal but should be excluded under these rules.
>
> *Why this matters:* fiscal vs. everything-else is the classification most prone to false positives — every government action sounds fiscal. The conservative rule trades a few missed cases for far fewer wrong ones.
---

## 5. Talking to the API

Three small helpers handle the mechanics. Read them as the "plumbing" — the economics lives in the prompts above; these just move data.

### 5.1 Sending a PDF
The Claude API accepts a PDF natively as a base64-encoded `document` block — the model reads the file directly, so we don't extract text ourselves. We pair the document with a short text instruction in one user message.

### 5.2 Getting clean JSON back
`extract_json` strips any stray markdown fences and parses. If parsing fails it returns the raw text under a flag rather than crashing, so one malformed reply doesn't kill a whole batch.

In [ ]:
def pdf_to_b64(path):
    """Read a PDF off disk and base64-encode it for the API."""
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def pdf_block(b64):
    """Wrap an encoded PDF as a Claude 'document' content block."""
    return {"type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": b64}}

def extract_json(text):
    """Parse the model's reply into a dict, tolerating markdown fences."""
    text = re.sub(r"^```(?:json)?", "", text.strip()).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)   # grab the first {...} block
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                pass
        return {"raw_response": text, "parse_error": True}

def call_model(content_blocks, system_prompt):
    """One API call: system prompt + user content -> parsed JSON dict."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1500,
        system=system_prompt,                 # the codebook goes here
        messages=[{"role": "user", "content": content_blocks}],
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    return extract_json(text)

print("API helpers defined.")

### 5.3 Bridging Stage 1 → Stage 2

This is where the two-stage design becomes code. `derive_context` reads Stage 1's output and decides what to hand Stage 2:

- If the driver is **Demand** (or **Both** with a usable demand sentiment) → tell Stage 2 "main driver = Demand, sentiment = X".
- Else if **Supply** → hand over the supply side.
- If neither is usable (**Undeterminable**, or sentiment is **Null**) → there is nothing to refine, so we *skip* Stage 2 entirely.

The "prefer demand on Both" choice is a simplification: the subtype schema records a single main driver per row. If you wanted both sides of a "Both drivers" article subtyped separately, this is exactly the function you'd extend.

In [ ]:
def derive_context(primary):
    """From Stage 1 output, pick the (main driver, sentiment) to feed Stage 2.
    Returns None when there is nothing classifiable to refine."""
    drivers = primary.get("Drivers of sentiment", "")
    sent = primary.get("Sentiment", {}) or {}
    dem, sup = sent.get("Demand drivers"), sent.get("Supply drivers")

    if drivers == "Demand drivers" or (drivers == "Both drivers" and dem not in (None, "Null")):
        return {"main": "Demand drivers", "sentiment": dem}
    if drivers == "Supply drivers" or (drivers == "Both drivers" and sup not in (None, "Null")):
        return {"main": "Supply drivers", "sentiment": sup}
    return None

print("Stage-1 -> Stage-2 bridge defined.")

---
> ### 🔧 YOUR TURN
> `derive_context` is where the two-stage design lives. Right now, for a **"Both drivers"** article it always prefers the *demand* side and sends only that to Stage 2.
>
> 1. Trace by hand: given `{"Drivers of sentiment": "Both drivers", "Sentiment": {"Demand drivers": "Neutral", "Supply drivers": "Positive"}}`, what does the function return? Is that the behaviour you'd want?
> 2. **Stretch:** sketch (in comments or pseudocode) how you'd change the pipeline so that a "Both drivers" article gets *both* its demand and supply sides subtyped and written to the row, instead of just one.
>
> *Why this matters:* this is the one place the code makes an economic simplification. Knowing where your method cuts a corner is part of doing the method honestly.
---

## 6. Running the full classification

This loop ties it together: for every PDF, for both tasks, run Stage 1, then conditionally Stage 2. The results nest as:

```
{ filename: { "growth":    {"primary": {...}, "subtype": {...|None}},
              "inflation": {"primary": {...}, "subtype": {...|None}} } }
```

> **⚠ This cell makes live API calls** — 4 PDFs × 2 tasks × up to 2 stages ≈ 16 calls. It costs money and takes a minute or two.

### A note on saving: the encoding gotcha
We save with `encoding="utf-8"` **explicitly**. On Windows, Python's default text encoding is *cp1252*, and an em-dash (—) in a model's explanation would be written as a byte that later fails to decode as UTF-8. Forcing UTF-8 on write — and tolerating cp1252 on read (Section 8) — avoids a `UnicodeDecodeError` that otherwise bites only on Windows.

In [ ]:
def run_classification():
    results = {}
    for fname in PDF_FILES:
        path = PDF_DIR / fname
        if not path.exists():
            print("  WARNING: not found, skipping:", path)
            continue
        b64 = pdf_to_b64(path)
        results[fname] = {}

        for task in ("growth", "inflation"):
            print(f"{fname} [{task}] - stage 1 ...")
            primary = call_model(
                [pdf_block(b64),
                 {"type": "text", "text": "Classify the article. Respond with only the JSON object."}],
                PRIMARY_PROMPTS[task],
            )
            entry = {"primary": primary, "subtype": None}

            ctx = derive_context(primary)
            if ctx and ctx["sentiment"] not in (None, "Null"):
                print(f"{fname} [{task}] - stage 2 ...")
                ctx_text = (f"Input context for this article:\n"
                            f"- Sentiment: {ctx['sentiment']}\n"
                            f"- Main drivers: {ctx['main']}\n\n"
                            f"Classify the driver subtypes. Respond with only the JSON object.")
                entry["subtype"] = call_model(
                    [pdf_block(b64), {"type": "text", "text": ctx_text}],
                    SUBTYPE_PROMPTS[task],
                )
            else:
                print(f"   skip stage 2 [{task}] (nothing classifiable)")
            results[fname][task] = entry

    with open(RESULTS_JSON, "w", encoding="utf-8") as f:   # explicit UTF-8!
        json.dump(results, f, indent=2, ensure_ascii=False)
    print("Saved ->", RESULTS_JSON)
    return results

# Uncomment to run the live classification:
# results = run_classification()

---
> ### 🔧 YOUR TURN  *(uses live API — costs a few calls)*
> Time to actually run it.
>
> 1. Uncomment `results = run_classification()` and run it on your four PDFs. Watch the printout: which articles **skipped Stage 2**, and why?
> 2. Open `classification_results.json` and read the `explanation` field for one article. Do you agree with the model's reasoning? Find the single classification you most disagree with.
> 3. Re-run `run_classification()` a second time on the same PDFs. Did any label change? (This is a quick, informal **reliability** check.)
>
> *Why this matters:* an LLM coder is only useful if you've looked at its work. The `explanation` field is your audit trail — use it.
---

## 7. From classifications to the spreadsheet

### The economics of the layout
The authors' spreadsheet has one row per article and a column per subtype. The article's **sentiment score** (+1/−1/0) is placed in the column matching its **subtype**. Three aggregate columns then sum across:

- **Demand total** = sum of the four demand columns
- **Supply total** = sum of the four supply columns
- **Total** = Demand + Supply

Column map (matching the published file exactly):

| Col | Meaning | | Col | Meaning |
|---|---|---|---|---|
| B | Real · Private | | G | Commodity prices |
| C | Real · Fiscal Policy | | H | Supply disruptions |
| D | Financial · Private | | I | Government Policy |
| E | Financial · Monetary Policy | | J | Others |

### The code mechanics: formulas, not hardcoded sums
We write the aggregate columns as **Excel formulas** (`=SUM(B9:E9)`), not pre-computed numbers. That keeps the sheet live — if you edit a score by hand, the totals update. This is standard spreadsheet hygiene.

In [ ]:
SENT = {"positive": 1, "negative": -1, "neutral": 0}

DEMAND_COL = {
    "real, private drivers": "B",
    "real, fiscal policy drivers": "C",
    "financial, private drivers": "D",
    "financial, monetary policy drivers": "E",
}
SUPPLY_COL = {
    "commodity prices": "G",
    "supply disruptions": "H",
    "policy shifts": "I",        # the spreadsheet labels this "Government Policy"
    "others": "J",
}

HEADER_TITLE = {
    "growth":    "Macroeconomic sentiment indices and detailed drivers,  (a) Growth sentiment and drivers",
    "inflation": "Macroeconomic sentiment indices and detailed drivers,  (b) Inflation sentiment and drivers",
}

def sentiment_score(primary, side):
    s = (primary.get("Sentiment", {}) or {}).get(side)
    return None if s in (None, "Null") else SENT.get(str(s).strip().lower())

def subtype_string(subtype, side):
    if not subtype:
        return None
    ds = subtype.get("Driver subtypes", {}) or {}
    if side == "Demand drivers":
        v = ds.get("Demand drivers")
        return None if v in (None, "Null") else str(v).strip().lower()
    v = (ds.get("Supply drivers", {}) or {}).get("Type")
    return None if v in (None, "Null") else str(v).strip().lower()

def build_row(entry):
    """Map one article's result to {column_letter: score}."""
    primary, subtype = entry.get("primary", {}) or {}, entry.get("subtype")
    cells = {}
    for side, colmap in (("Demand drivers", DEMAND_COL), ("Supply drivers", SUPPLY_COL)):
        score = sentiment_score(primary, side)
        if score is None:
            continue
        st = subtype_string(subtype, side)
        col = colmap.get(st) if st else None
        if col is None:               # sentiment present but no subtype -> fallback
            col = "B" if side == "Demand drivers" else "J"
        cells[col] = cells.get(col, 0) + score
    return cells

print("Mapping helpers defined.")

---
> ### 🔧 YOUR TURN
> `build_row` maps each article to a column. Note the **fallback**: if Stage 1 found a sentiment but Stage 2 returned no subtype, the score is parked in Real-Private (demand) or Others (supply) rather than dropped.
>
> 1. With only four articles, check whether *any* of yours hit that fallback (look for a non-zero score in column B or J whose `subtype` JSON is `Null`).
> 2. Decide what *you* think is the right behaviour: silently park it, drop it, or flag it for manual review? Change one line of `build_row` to do whatever you chose.
>
> *Why this matters:* every pipeline has edge cases. The danger isn't having them — it's handling them invisibly so they distort your totals without you noticing.
---

The fallback in `build_row` deserves a comment: if Stage 1 found a sentiment but Stage 2 couldn't pin a subtype, we still record the score (in Real-Private for demand, Others for supply) so the article isn't silently dropped from the totals. With only four articles you can eyeball any such case against the JSON `explanation` and move it by hand.

Below, `write_sheet` lays out the two-tier header and one row per article, and `export_excel` builds both sheets.

In [ ]:
def write_sheet(ws, task, results, date_label):
    ws["A1"] = HEADER_TITLE[task]; ws["A1"].font = Font(bold=True)
    ws["B6"] = "Demand drivers"; ws["G6"] = "Supply drivers"
    ws["L6"] = "Demand drivers"; ws["M6"] = "Supply drivers"; ws["O6"] = "Total"
    ws["B7"] = "Real"; ws["D7"] = "Financial"
    ws["B8"] = "Private"; ws["C8"] = "Fiscal Policy"
    ws["D8"] = "Private"; ws["E8"] = "Monetary Policy"
    ws["G8"] = "Commodity prices"; ws["H8"] = "Supply disruptions"
    ws["I8"] = "Government Policy"; ws["J8"] = "Others"
    for c in ("B6","G6","L6","M6","O6","B7","D7"): ws[c].font = Font(bold=True)
    for col in "BCDEGHIJ": ws[f"{col}8"].font = Font(bold=True)

    r = 9
    for fname in sorted(results):
        entry = results[fname].get(task)
        if not entry:
            continue
        ws.cell(row=r, column=1, value=date_label)
        for col, val in build_row(entry).items():
            ws[f"{col}{r}"] = val
        ws[f"L{r}"] = f"=SUM(B{r}:E{r})"     # demand total (formula)
        ws[f"M{r}"] = f"=SUM(G{r}:J{r})"     # supply total (formula)
        ws[f"O{r}"] = f"=L{r}+M{r}"          # grand total (formula)
        r += 1

def export_excel(results):
    wb = Workbook(); wb.remove(wb.active)
    for task in ("growth", "inflation"):
        write_sheet(wb.create_sheet(task), task, results, DATE_LABEL)
    wb.save(OUTPUT_XLSX)
    print("Saved ->", OUTPUT_XLSX)

print("Excel writers defined.")

## 8. Putting it together

If you already ran the classification (Section 6) and have `results` in memory, just call `export_excel(results)`.

If you're returning later and only have the saved JSON, load it first. The loader tries UTF-8 and falls back to cp1252 — the read-side companion to the write-side fix from Section 6.

In [ ]:
def load_results_tolerant(path):
    """Read the results JSON, falling back to cp1252 if it isn't valid UTF-8."""
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except UnicodeDecodeError:
        with open(path, encoding="cp1252") as f:
            return json.load(f)

# Typical end-to-end use:
#   results = run_classification()      # Section 6 (live API)
#   export_excel(results)               # build the spreadsheet
#
# Or, rebuild the Excel from a saved run without spending API calls:
#   results = load_results_tolerant(RESULTS_JSON)
#   export_excel(results)
print("Ready. Uncomment the calls above to run end-to-end.")

## 9. Recap and extensions

**What you built.** A reproducible pipeline that reads news PDFs and produces the BIS WP 1294 sentiment table. The economics drove every design choice; the code just executed it faithfully.

**The ideas worth keeping:**
- *Demand vs. supply* is identifiable from text because the two have opposite signatures in activity vs. inflation.
- *Decomposing* a hard classification into conditional stages improves both accuracy and auditability — Stage 2 trusts Stage 1's verdict and only refines it.
- *The prompt is the codebook.* Reproducibility comes from writing the coding rules out explicitly, including conservative tie-breaks (the fiscal-policy rule).
- *Structured output* turns an LLM into a measurement instrument: fixed schema in, parsed JSON out.
- *Boring-but-real engineering* — UTF-8 encoding, formulas instead of hardcoded sums, defensive parsing — is what makes the result trustworthy.

**Ways to extend this for a project:**
1. **Many articles, real dates.** Add a publication date to each result and aggregate multiple articles per day, as the authors do, to build a genuine time series.
2. **Confidence filtering.** Each call returns a confidence; drop or flag low-confidence classifications and see how the index changes.
3. **Validation.** Hand-code a sample yourself and measure agreement with the model — the empirical core of any LLM-as-coder paper.
4. **Both-drivers handling.** Extend `derive_context` to subtype *both* sides of a "Both drivers" article rather than preferring demand.
5. **Reliability.** Run each article a few times and check how stable the labels are.

> *Sources:* Kwon, B., T. Park, P. Rungcharoenkitkul, and F. Smets (2025), "Parsing the pulse: decomposing macroeconomic sentiment with LLMs", BIS Working Paper No. 1294. Article data: Dow Jones Factiva / Wall Street Journal.